# 07 — Statistiques du jeu de détection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/07_stats_yolo.ipynb)

Vue statistique de `data/yolo/` (docs/19 §4) : effectifs par split, source et classe,
géométrie des boîtes (petits objets, cadrage), qualité des annotations (doublons, boîtes
imbriquées, dégénérées), fuite de groupes, planches d'exemples. Calcul dans
`vision/datasets/stats.py` ; sorties dans `reports/datasets/yolo/`. CPU seulement.

## Paramètres

In [ ]:
IMG_SIZE = 640  # entrée du détecteur (letterbox : grand côté -> IMG_SIZE)
N_EXAMPLES = 8  # images par planche
SEED = 0
SOURCE = None  # planches : "plantdoc" ou "plantvillage" ; None : toutes
PV_PER_CLASS = 40  # si data/yolo/ doit être reconstruit (docs/04 §2)
DATASETS = ["index"]  # data/yolo/ : jeu `yolo` du Drive, sinon reconstruit ci-dessous

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

## Jeu `data/yolo/`

In [ ]:
import pandas as pd
import yaml
from IPython.display import Markdown, display

from common.report import REPORTS_DIR, write_report
from vision.datasets import stats as st
from vision.datasets import stats_plots as sp
from vision.datasets.build_index import DATA_DIR, INDEX_NAME
from vision.labeling import build_yolo_dataset as byd

YOLO_DIR = DATA_DIR / byd.OUT_NAME
try:
    drive_data.ensure(["yolo"], DRIVE_DIR if COLAB else None)
except RuntimeError as e:
    print(e, "\n-> reconstruction depuis PlantVillage et PlantDoc")
    drive_data.ensure(["plantvillage", "plantdoc"], DRIVE_DIR if COLAB else None)
    byd.build(DATA_DIR, YOLO_DIR, PV_PER_CLASS, os.cpu_count())

OUT = REPORTS_DIR / "yolo"
OUT.mkdir(parents=True, exist_ok=True)
manifest = pd.read_parquet(YOLO_DIR / byd.MANIFEST)
names = yaml.safe_load((YOLO_DIR / "data.yaml").read_text())["names"]
NAMES = [names[i] for i in sorted(names)]
boxes = st.read_yolo_labels(YOLO_DIR, manifest)
counts = st.yolo_counts(manifest, boxes, NAMES)
SUMMARY = {"n_images": counts["n_images"], "n_boites": counts["n_boites"],
           "classes": NAMES, "classes_sans_annotation": counts["classes_sans_annotation"],
           "images_sans_boite": len(counts["images_sans_boite"])}
SECTIONS = {}

In [ ]:
# À lire avant toute métrique par classe du détecteur.
empty = counts["classes_sans_annotation"]
if empty:
    display(Markdown(
        f"> ⚠️ **{len(empty)} classe(s) sans aucune annotation : {', '.join(empty)}.** "
        f"Elles sont déclarées dans `data.yaml`, mais seules les métriques de "
        f"{', '.join(n for n in NAMES if n not in empty)} ont un sens. "
        f"{SUMMARY['images_sans_boite']} image(s) sans boîte."
    ))
display(pd.Series(SUMMARY).to_frame("valeur"))

## 4.1 Effectifs

In [ ]:
print("images source × split")
display(counts["images"])
print("boîtes source × split")
display(counts["boites"])
display(counts["boites_par_classe"])
print(f"{len(counts['images_sans_boite'])} images sans boîte (négatifs)")
display(counts["images_sans_boite"].head(20))
neg = counts["images_sans_boite"].head(N_EXAMPLES)
if len(neg):
    display(sp.plot_image_grid([YOLO_DIR / p for p in neg["image"]], list(neg["source"]),
                               cols=4, path=OUT / "sans_boite.png"))
bad = st.nboxes_consistency(manifest, boxes)
print(f"n_boxes du manifest ≠ lignes de labels/ : {len(bad)} images (attendu : 0)")
display(bad.head(20))
SUMMARY["incoherences_n_boxes"] = len(bad)
SECTIONS["Images source × split"] = counts["images"]
SECTIONS["Boîtes source × split"] = counts["boites"]

## 4.2 Géométrie des boîtes

Tailles à l'entrée du réseau (`IMG_SIZE`) : catégories COCO small (< 32²), medium
(< 96²), large. La mAP chute sur les petits objets : leur part guide le choix de la
taille d'entrée (640 vs 960) ou du tuilage.

In [ ]:
geom = st.box_geometry(boxes, IMG_SIZE)
per_img = st.boxes_per_image(manifest, boxes)
display(per_img.groupby("source")["n_lignes"].describe())
display(geom.groupby("source")[["aire_rel", "w_px", "h_px", "aspect"]].median())
coco = st.coco_share(geom)
display(coco)
display(sp.plot_box_geometry(geom, OUT / "geometrie_boites.png"))
maps = {s: st.center_heatmap(g) for s, g in geom.groupby("source")}
display(sp.plot_heatmaps(maps, OUT / "centres.png"))
SUMMARY.update(part_coco=coco.to_dict(orient="index"),
               boites_par_image=per_img.groupby("source")["n_lignes"].mean().to_dict())
SECTIONS["Catégories COCO à IMG_SIZE"] = coco

## 4.3 Qualité des annotations

Doublons probables (IoU > 0,9), boîtes imbriquées, boîtes dégénérées (< 2 px à
`IMG_SIZE`) ou débordant de l'image ; fuite de groupes entre splits. Le manifest n'a
pas de `groupe` pour PlantVillage : il est repris de `index.parquet` par `path`.

In [ ]:
q = st.box_quality(boxes, IMG_SIZE)
display(pd.Series({k: len(v) for k, v in q.items()}, name="n").to_frame())
for k in ["doublons", "imbriquees"]:
    if len(q[k]):
        print(k)
        display(q[k].head(10))
index = pd.read_parquet(DATA_DIR / INDEX_NAME, columns=["path", "groupe"])
leaks = st.yolo_group_leaks(manifest, index)
print(f"groupes dans plusieurs splits : {leaks['n_fuites']} (attendu : 0) ; "
      f"images non contrôlées : {leaks['non_controlees']}",
      leaks["non_controlees_par_source"])
display(leaks["fuites"].head(10))
SUMMARY.update(qualite={k: len(v) for k, v in q.items()},
               fuites_groupes=leaks["n_fuites"], non_controlees=leaks["non_controlees"])

## 4.4 Exemples

In [ ]:
sel = manifest if SOURCE is None else manifest[manifest["source"] == SOURCE]
for split in ["train", "val", "test"]:
    sub = sel[sel["split"] == split]
    if len(sub):
        print(split)
        display(sp.plot_yolo_examples(YOLO_DIR, sub, boxes, N_EXAMPLES, SEED,
                                      OUT / f"exemples_{split}.png"))

## Rapport

In [ ]:
write_report(OUT, SUMMARY, SECTIONS)
drive_data.sync_all(DRIVE_DIR)
display(Markdown((OUT / "summary.md").read_text(encoding="utf-8")))

## Résumé

`reports/datasets/yolo/` : `summary.json`, `summary.md`, figures. Attendu (docs/19
§4) : 4 427 images, 10 743 boîtes, toutes `feuille`. Décisions (docs/19 §9) : retirer
`plante` et `fruit` de `data.yaml` ou lancer leur annotation ; taille d'entrée selon la
part de petites boîtes.